# Nguyễn Hoàng Trọng Sơn - 22521252

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, avg, count, when, year, from_unixtime

In [2]:
spark = SparkSession.builder.getOrCreate()
movies_df = spark.read.csv("/content/movies.txt", sep=",", header=False).select(
col("_c0").cast("int").alias("MovieID"),
col("_c1").alias("Title"),
col("_c2").alias("Genres")
)
ratings1_df = spark.read.csv("/content/ratings_1.txt", sep=",", header=False).select(
col("_c0").cast("int").alias("UserID"),
col("_c1").cast("int").alias("MovieID"),
col("_c2").cast("float").alias("Rating"),
col("_c3").cast("long").alias("Timestamp")
)
ratings2_df = spark.read.csv("/content/ratings_2.txt", sep=",", header=False).select(
col("_c0").cast("int").alias("UserID"),
col("_c1").cast("int").alias("MovieID"),
col("_c2").cast("float").alias("Rating"),
col("_c3").cast("long").alias("Timestamp")
)

# Bài 1: Tính Điểm Đánh Giá Trung Bình và Tổng Số Lượt Đánh Giá Cho Mỗi Phim

## Mục tiêu
- Tính điểm trung bình cho từng phim từ cả 2 file ratings (ratings_1.txt và ratings_2.txt).
- Tính tổng số lượt đánh giá cho mỗi phim.
- Tìm ra phim có điểm trung bình cao nhất (chỉ xét những phim có tối thiểu 5 lượt đánh giá).
## Output:
- MovieTitle AverageRating: xx (TotalRatings: xx)
- MovieTitle is the highest rated movie with an average rating of AverageRating among movies with at least 5 ratings.

In [3]:
all_ratings_df = ratings1_df.union(ratings2_df)
agg_df = all_ratings_df.groupBy("MovieID").agg(avg("Rating").alias("AvgRating"), count("Rating").alias("TotalRatings"))
joined_df = agg_df.join(movies_df, "MovieID")
filtered_df = joined_df.filter(col("TotalRatings") >= 5)
sorted_df = filtered_df.orderBy(col("AvgRating").desc())
results = sorted_df.collect()
for row in results:
    print(f"{row['Title']} AverageRating: {round(row['AvgRating'], 2)}␣(TotalRatings: {row['TotalRatings']})")
if len(results) > 0:
    top_movie = results[0]
    print(f"\n{top_movie['Title']} is the highest rated movie with an average␣rating of {round(top_movie['AvgRating'], 2)} among movies with at least 5␣ratings.")
else:
    print("No movies found with at least 5 ratings.")

Sunset Boulevard (1950) AverageRating: 4.36␣(TotalRatings: 7)
The Terminator (1984) AverageRating: 4.06␣(TotalRatings: 18)
The Godfather: Part II (1974) AverageRating: 4.0␣(TotalRatings: 17)
The Lord of the Rings: The Fellowship of the Ring (2001) AverageRating: 3.89␣(TotalRatings: 18)
No Country for Old Men (2007) AverageRating: 3.89␣(TotalRatings: 18)
The Social Network (2010) AverageRating: 3.86␣(TotalRatings: 7)
The Lord of the Rings: The Return of the King (2003) AverageRating: 3.82␣(TotalRatings: 11)
E.T. the Extra-Terrestrial (1982) AverageRating: 3.67␣(TotalRatings: 18)
Gladiator (2000) AverageRating: 3.61␣(TotalRatings: 18)
Fight Club (1999) AverageRating: 3.5␣(TotalRatings: 7)
Mad Max: Fury Road (2015) AverageRating: 3.47␣(TotalRatings: 18)
Lawrence of Arabia (1962) AverageRating: 3.44␣(TotalRatings: 18)
The Silence of the Lambs (1991) AverageRating: 3.14␣(TotalRatings: 7)

Sunset Boulevard (1950) is the highest rated movie with an average␣rating of 4.36 among movies with at 

# Bài 2: Phân Tích Đánh Giá Theo Thể Loại
## Mục tiêu:
- Vì một phim có thể thuộc nhiều thể loại (Genres được phân tách bằng dấu “|”), mapper cần tách riêng từng thể loại của phim đó.
- Tính điểm trung bình (và tổng số lượt đánh giá) cho từng thể loại, dựa trên tất cả các phim thuộc thể loại đó.
## Output: Genre - AverageRating (TotalRatings)

In [4]:
joined_df = all_ratings_df.join(movies_df, "MovieID")

split_genre_df = joined_df.withColumn("Genre", explode(split(col("Genres"),r"\|")))
genre_agg_df = split_genre_df.groupBy("Genre").agg(
    avg("Rating").alias("AvgRating"),
    count("Rating").alias("TotalRatings")
)

genre_agg_df = genre_agg_df.orderBy(col("AvgRating").desc())
results = genre_agg_df.collect()

for row in results:
    print(f"{row['Genre']} - AverageRating: {round(row['AvgRating'], 2)}(TotalRatings: {row['TotalRatings']})")

Film-Noir - AverageRating: 4.36(TotalRatings: 7)
Mystery - AverageRating: 4.0(TotalRatings: 2)
Horror - AverageRating: 4.0(TotalRatings: 2)
Fantasy - AverageRating: 3.86(TotalRatings: 29)
Crime - AverageRating: 3.81(TotalRatings: 42)
Drama - AverageRating: 3.76(TotalRatings: 128)
Sci-Fi - AverageRating: 3.73(TotalRatings: 54)
Action - AverageRating: 3.71(TotalRatings: 54)
Thriller - AverageRating: 3.7(TotalRatings: 27)
Family - AverageRating: 3.67(TotalRatings: 18)
Adventure - AverageRating: 3.63(TotalRatings: 83)
Biography - AverageRating: 3.56(TotalRatings: 25)


# Bài 3: Phân Tích Đánh Giá Theo Giới Tính
## Mục tiêu:
- Thực hiện join dữ liệu giữa ratings và users (dựa trên UserID) để lấy thông tin giới tính của người đánh giá.
- Với mỗi phim, tính riêng điểm trung bình từ người dùng nam và nữ.
## Output: MovieTitle - Male_Avg: xx, Female_Avg: xx

In [5]:
users_df = spark.read.csv("/content/users.txt", sep=",", header=False) \
    .select(col("_c0").cast("int").alias("UserID"), col("_c1").alias("Gender"))

In [6]:
ratings_with_users_df = all_ratings_df.join(users_df, "UserID")
ratings_with_users_movies_df = ratings_with_users_df.join(movies_df, "MovieID")

movie_gender_agg_df = ratings_with_users_movies_df.groupBy("Title") \
    .agg(avg(when(col("Gender") == "M", col("Rating"))).alias("Male_Avg"),
        avg(when(col("Gender") == "F", col("Rating"))).alias("Female_Avg"))

results = movie_gender_agg_df.collect()

for row in results:
    male_avg = round(row["Male_Avg"], 2) if row["Male_Avg"] is not None else "None"
    female_avg = round(row["Female_Avg"], 2) if row["Female_Avg"] is not None else "None"
    print(f"{row['Title']} - Male_Avg: {male_avg}, Female_Avg: {female_avg}")

Psycho (1960) - Male_Avg: None, Female_Avg: 4.0
No Country for Old Men (2007) - Male_Avg: 3.92, Female_Avg: 3.83
E.T. the Extra-Terrestrial (1982) - Male_Avg: 3.81, Female_Avg: 3.55
Gladiator (2000) - Male_Avg: 3.59, Female_Avg: 3.64
Fight Club (1999) - Male_Avg: 3.5, Female_Avg: 3.5
The Social Network (2010) - Male_Avg: 4.0, Female_Avg: 3.67
The Silence of the Lambs (1991) - Male_Avg: 3.33, Female_Avg: 3.0
The Terminator (1984) - Male_Avg: 3.93, Female_Avg: 4.14
The Lord of the Rings: The Fellowship of the Ring (2001) - Male_Avg: 4.0, Female_Avg: 3.8
Lawrence of Arabia (1962) - Male_Avg: 3.55, Female_Avg: 3.31
The Godfather: Part II (1974) - Male_Avg: 4.06, Female_Avg: 3.94
Mad Max: Fury Road (2015) - Male_Avg: 4.0, Female_Avg: 3.32
Sunset Boulevard (1950) - Male_Avg: 4.33, Female_Avg: 4.5
The Lord of the Rings: The Return of the King (2003) - Male_Avg: 3.75, Female_Avg: 3.9


# Bài 4: Phân Tích Đánh Giá Theo Nhóm Tuổi
## Mục tiêu:
- Phân nhóm người dùng theo độ tuổi (ví dụ: 0-18, 18-35, 35-50, 50+).
- Với mỗi phim, tính điểm trung bình cho mỗi nhóm tuổi.
## Output: MovieTitle - [0-18: AvgRating, 18-35: AvgRating, 35-50: AvgRating, 50+: AvgRating]

In [7]:
users_df = spark.read.csv("/content/users.txt", sep=",", header=False) \
    .select(col("_c0").cast("int").alias("UserID"),
            col("_c1").alias("Gender"),
            col("_c2").cast("int").alias("Age"))

In [8]:
users_with_age = users_df.withColumn("AgeGroup", when(col("Age") <= 18, "0–18") \
    .when((col("Age") > 18) & (col("Age") <= 35), "18–35") \
    .when((col("Age") > 35) & (col("Age") <= 50), "35–50") \
    .otherwise("50+"))

ratings_users_df = all_ratings_df.join(users_with_age, "UserID")
ratings_users_movies_df = ratings_users_df.join(movies_df, "MovieID")

pivot_df = ratings_users_movies_df.groupBy("Title") \
    .pivot("AgeGroup", ["0–18", "18–35", "35–50", "50+"]) \
    .agg(avg("Rating"))

results = pivot_df.collect()

for row in results:
    group0_18 = round(row["0–18"], 2) if row["0–18"] is not None else "None"
    group18_35 = round(row["18–35"], 2) if row["18–35"] is not None else "None"
    group35_50 = round(row["35–50"], 2) if row["35–50"] is not None else "None"
    group50plus = round(row["50+"], 2) if row["50+"] is not None else "None"
    print(f"{row['Title']} – 0–18: {group0_18}, 18–35: {group18_35}, 35–50: {group35_50}, 50+: {group50plus}")

Psycho (1960) – 0–18: None, 18–35: 4.5, 35–50: 3.5, 50+: None
No Country for Old Men (2007) – 0–18: None, 18–35: 3.81, 35–50: 3.94, 50+: 4.0
E.T. the Extra-Terrestrial (1982) – 0–18: None, 18–35: 3.56, 35–50: 3.83, 50+: 3.0
Gladiator (2000) – 0–18: None, 18–35: 3.44, 35–50: 3.81, 50+: 3.5
Fight Club (1999) – 0–18: None, 18–35: 3.5, 35–50: 3.5, 50+: 3.5
The Social Network (2010) – 0–18: None, 18–35: 4.0, 35–50: 3.67, 50+: None
The Silence of the Lambs (1991) – 0–18: None, 18–35: 3.0, 35–50: 3.25, 50+: None
The Terminator (1984) – 0–18: None, 18–35: 4.17, 35–50: 4.05, 50+: 3.75
The Lord of the Rings: The Fellowship of the Ring (2001) – 0–18: None, 18–35: 4.0, 35–50: 3.83, 50+: None
The Lord of the Rings: The Return of the King (2003) – 0–18: None, 18–35: 3.83, 35–50: 3.81, 50+: None
Lawrence of Arabia (1962) – 0–18: None, 18–35: 3.6, 35–50: 3.29, 50+: 4.5
The Godfather: Part II (1974) – 0–18: None, 18–35: 3.78, 35–50: 4.25, 50+: None
Mad Max: Fury Road (2015) – 0–18: None, 18–35: 3.36, 3

# Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng
## Mục tiêu: Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation.
## Output: Occupation - TotalRatings: xx, AverageRating: xx

In [9]:
occupation_df = spark.read.csv("/content/occupation.txt", sep=",", header=False) \
    .select(col("_c0").cast("int").alias("OccupationID"),
            col("_c1").alias("OccupationName"))

users_df = spark.read.csv("/content/users.txt", sep=",", header=False) \
    .select(col("_c0").cast("int").alias("UserID"),
            col("_c1").alias("Gender"),
            col("_c2").cast("int").alias("Age"),
            col("_c3").cast("int").alias("Occupation"))

In [10]:
users_with_occ_df = users_df.join(occupation_df, users_df["Occupation"] == occupation_df["OccupationID"], "left") \
    .drop("Occupation", "OccupationID") \
    .withColumnRenamed("OccupationName", "Occupation")

ratings_users_df = all_ratings_df.join(users_with_occ_df, "UserID")

occupation_agg_df = ratings_users_df.groupBy("Occupation") \
    .agg(avg("Rating").alias("AverageRating"), count("Rating").alias("TotalRatings"))

results = occupation_agg_df.collect()

for row in results:
    print(f"{row['Occupation']} - AverageRating: {round(row['AverageRating'], 2)} (TotalRatings: {row['TotalRatings']})")


Student - AverageRating: 4.0 (TotalRatings: 8)
Nurse - AverageRating: 3.86 (TotalRatings: 11)
Salesperson - AverageRating: 3.65 (TotalRatings: 17)
Lawyer - AverageRating: 3.65 (TotalRatings: 17)
Teacher - AverageRating: 3.7 (TotalRatings: 5)
Designer - AverageRating: 4.0 (TotalRatings: 13)
Consultant - AverageRating: 3.86 (TotalRatings: 14)
Programmer - AverageRating: 4.25 (TotalRatings: 10)
Artist - AverageRating: 3.73 (TotalRatings: 11)
Journalist - AverageRating: 3.85 (TotalRatings: 17)
Doctor - AverageRating: 3.69 (TotalRatings: 21)
Engineer - AverageRating: 3.56 (TotalRatings: 18)
Accountant - AverageRating: 3.58 (TotalRatings: 6)
Manager - AverageRating: 3.47 (TotalRatings: 16)


# Bài 6: Phân Tích Đánh Giá Theo Thời Gian
## Mục tiêu: Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.
## Output: Year - TotalRatings: xx, AverageRating: xx

In [11]:
ratings_with_year_df = all_ratings_df.withColumn("Year",
    year(from_unixtime(col("Timestamp"))))
year_agg_df = ratings_with_year_df.groupBy("Year") \
    .agg(count("Rating").alias("TotalRatings"), avg("Rating").alias("AverageRating"))
year_agg_results = year_agg_df.orderBy("Year").collect()

for row in year_agg_results:
    print(f"{row['Year']} - TotalRatings: {row['TotalRatings']}, AverageRating: {round(row['AverageRating'], 2)}")


2020 - TotalRatings: 184, AverageRating: 3.75
